**Setup toàn bộ môi trường + dataset + code**

In [1]:
# =========================================================
# CELL 1 - FULL SETUP COLAB
# =========================================================
# ===== ĐI VỀ /content =====
%cd /content
# ===== CHECK GPU =====
import torch

print("=" * 50)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("=" * 50)

# =========================================================
# CLONE GITHUB
# =========================================================

!git clone -b attention-unet2D https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git

# =========================================================
# DOWNLOAD DATASET ZIP TỪ GOOGLE DRIVE
# =========================================================

!gdown --id 1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW

# =========================================================
# UNZIP DATASET
# =========================================================

!unzip -q dataset_BTXRD.zip

# =========================================================
# KIỂM TRA DATASET
# =========================================================

print("\nDATASET STRUCTURE:")
!ls dataset_BTXRD

# =========================================================
# COPY DATASET VÀO PROJECT
# =========================================================

!mv dataset_BTXRD Prompt-Guided-XRay-Segmentation/

# =========================================================
# ĐI VÀO PROJECT
# =========================================================

%cd Prompt-Guided-XRay-Segmentation

# =========================================================
# INSTALL REQUIREMENTS
# =========================================================

!pip install -q tqdm opencv-python matplotlib scikit-image gdown

print("\nSETUP DONE!")

/content
CUDA: True
GPU: Tesla T4
Cloning into 'Prompt-Guided-XRay-Segmentation'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 158 (delta 50), reused 65 (delta 27), pack-reused 62 (from 1)
Receiving objects: 100% (158/158), 36.61 MiB | 38.96 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW
From (redirected): https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW&confirm=t&uuid=be489a27-0433-4337-831e-5f62dd4d1d78
To: /content/dataset_BTXRD.zip
100% 1.51G/1.51G [00:15<00:00, 97.2MB/s]

DATASET STRUCTURE:
test  train  val
/content/Prompt-Guided-XRay-Segmentati

In [ ]:
%%writefile train_attunet.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import logging
import datetime

from dataset_simple import BTXRD_Dataset
from models.networks.attention_unet_2D import Attention_UNet_2D

# =========================================================
# CAU HINH — CHOT CUOI (CONG BANG VOI UNET & PGA)
# =========================================================
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE    = 4
EPOCHS        = 100
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
IMG_SIZE      = 512
PATIENCE      = 15
SCHED_PATIENCE = 5


def dice_loss(pred, target, smooth=1e-5):
    pred_soft    = torch.sigmoid(pred)
    intersection = (pred_soft * target).sum(dim=(1, 2, 3))
    union        = pred_soft.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return (1 - (2. * intersection + smooth) / (union + smooth)).mean()


def batch_metrics_sum(pred, target, smooth=1e-5):
    pred_bin = (torch.sigmoid(pred) > 0.5).float()
    tp = (pred_bin * target).sum(dim=(1, 2, 3))
    fp = (pred_bin * (1 - target)).sum(dim=(1, 2, 3))
    fn = ((1 - pred_bin) * target).sum(dim=(1, 2, 3))
    dice      = (2. * tp + smooth) / (2. * tp + fp + fn + smooth)
    iou       = (tp + smooth) / (tp + fp + fn + smooth)
    precision = tp / (tp + fp + smooth)
    recall    = tp / (tp + fn + smooth)
    return dice.sum().item(), iou.sum().item(), precision.sum().item(), recall.sum().item()


def setup_logger():
    os.makedirs("logs", exist_ok=True)
    t = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    logging.basicConfig(
        level=logging.INFO, format='%(message)s',
        handlers=[
            logging.FileHandler(f"logs/train_attunet_{t}.log", encoding='utf-8'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger()


def main():
    logger = setup_logger()
    logger.info("=" * 90)
    logger.info(f"TRAIN ATTENTION U-NET 2D | Device: {DEVICE}")
    logger.info(f"Batch: {BATCH_SIZE} | MaxEpochs: {EPOCHS} | LR: {LR} | ImgSize: {IMG_SIZE}")
    logger.info(f"WeightDecay: {WEIGHT_DECAY} | EarlyStop patience: {PATIENCE}")
    logger.info("=" * 90)

    train_ds = BTXRD_Dataset(
        image_dir="dataset_BTXRD/train/images",
        mask_dir="dataset_BTXRD/train/masks",
        img_size=IMG_SIZE, is_train=True
    )
    val_ds = BTXRD_Dataset(
        image_dir="dataset_BTXRD/val/images",
        mask_dir="dataset_BTXRD/val/masks",
        img_size=IMG_SIZE, is_train=False
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model         = Attention_UNet_2D(in_channels=1, n_classes=1).to(DEVICE)
    criterion_bce = nn.BCEWithLogitsLoss()
    optimizer     = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler     = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=SCHED_PATIENCE, min_lr=1e-7)

    os.makedirs("checkpoints", exist_ok=True)
    best_val_dice = 0.0
    no_improve    = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for images, masks in loop:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            preds = model(images)
            loss  = criterion_bce(preds, masks) + dice_loss(preds, masks)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            loop.set_postfix(loss=f"{loss.item():.4f}")

        model.eval()
        sum_dice, sum_iou, sum_pre, sum_rec = 0, 0, 0, 0
        total = 0
        with torch.no_grad():
            for vi, vm in val_loader:
                vi, vm = vi.to(DEVICE), vm.to(DEVICE)
                vout   = model(vi)
                d, i, p, r = batch_metrics_sum(vout, vm)
                sum_dice += d; sum_iou += i; sum_pre += p; sum_rec += r
                total += vi.size(0)

        val_dice = sum_dice / total
        scheduler.step(val_dice)

        log_str = (f"Epoch {epoch+1:3d} | Loss: {train_loss/len(train_loader):.4f} | "
                   f"Dice: {val_dice:.4f} | IoU: {sum_iou/total:.4f} | "
                   f"Pre: {sum_pre/total:.4f} | Rec: {sum_rec/total:.4f} | "
                   f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        torch.save(model.state_dict(), "checkpoints/attunet_last.pth")
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            no_improve    = 0
            torch.save(model.state_dict(), "checkpoints/att_unet_best.pth")
            log_str = "[BEST] " + log_str
        else:
            no_improve += 1

        logger.info(log_str)

        if no_improve >= PATIENCE:
            logger.info(f"\nEarly stopping at epoch {epoch+1} (no improve for {PATIENCE} epochs)")
            break

    logger.info(f"\nBest Val Dice: {best_val_dice:.4f}")
    logger.info("Checkpoint: checkpoints/attunet_best.pth")


if __name__ == "__main__":
    main()


**Train**

In [2]:
# =========================================================
# TRAIN
# =========================================================

!python train_attunet.py


**Test + Dice + IoU + BCE**

ATT_Unet2D

In [3]:
# @title
# =========================================================
# TEST PHASE (ATTENTION U-NET 2D | ImgSize=512 | Batch=4 | Epochs≤100)
# =========================================================

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt

# Import model và dataset
from dataset_simple import BTXRD_Dataset
from models.networks.attention_unet_2D import Attention_UNet_2D

# =========================================================
# CẤU HÌNH
# =========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "checkpoints/att_unet_best.pth"
IMG_SIZE = 512  # Chú ý: File này của bạn dùng 512

# =========================================================
# HÀM HỖ TRỢ (LỌC NHIỄU & TÍNH TOÁN) - CHUẨN CHUNG
# =========================================================

def extract_lcc(binary_map: np.ndarray) -> np.ndarray:
    """Lọc sạch nhiễu viền, chỉ giữ lại cục xương to nhất để công bằng với các model khác."""
    if binary_map.sum() == 0:
        return binary_map
    mask_uint8 = binary_map.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_uint8, connectivity=8)
    if num_labels <= 1:
        return binary_map
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest_label).astype(np.float32)

def calc_hd95(pred: np.ndarray, gt: np.ndarray) -> float:
    """Tính 95% Hausdorff Distance"""
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred)
    ge = gt   ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pred_bin: np.ndarray, gt_bin: np.ndarray):
    """Tính độ lệch tâm Center-Based Localization"""
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2)
    return float(np.clip(1.0 - d/gt_diag, 0.0, 1.0))

def get_centroid(binary_map: np.ndarray):
    """Lấy tọa độ tâm để vẽ lên biểu đồ"""
    if binary_map.sum() == 0: return None, None
    ys, xs = np.where(binary_map)
    return float(xs.mean()), float(ys.mean())

# =========================================================
# LOAD MODEL & DATASET
# =========================================================

model = Attention_UNet_2D(in_channels=1, n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

test_dataset = BTXRD_Dataset(
    image_dir="dataset_BTXRD/test/images",
    mask_dir="dataset_BTXRD/test/masks",
    img_size=IMG_SIZE,
    is_train=False
)
test_dataset.images = sorted(test_dataset.images)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# =========================================================
# TEST LOOP
# =========================================================

SHOW_INDEX = list(range(10))  # 10 ảnh đầu theo thứ tự
all_dice, all_iou, all_pre, all_rec, all_hd95, all_cbl = [], [], [], [], [], []
smooth = 1e-5

with torch.no_grad():
    for idx, (images, masks) in enumerate(test_loader):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        outputs = model(images)
        preds = (torch.sigmoid(outputs) > 0.5).float()

        # Đưa về Numpy
        img_np = (images[0, 0].cpu().numpy() + 1) / 2.0  # Chỉnh lại cho sáng nếu ảnh bị tối do normalize
        gm = masks[0, 0].cpu().numpy()
        pm = preds[0, 0].cpu().numpy()

        # HẬU XỬ LÝ (Lọc nhiễu mảnh nhỏ để HD95 chuẩn xác)
        pm = extract_lcc(pm)

        # Tính 4 thông số cơ bản
        tp = (pm * gm).sum()
        fp = (pm * (1 - gm)).sum()
        fn = ((1 - pm) * gm).sum()

        dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
        iou  = (tp + smooth) / (tp + fp + fn + smooth)
        pre  = (tp + smooth) / (tp + fp + smooth)
        rec  = (tp + smooth) / (tp + fn + smooth)

        # Tính khoảng cách & độ lệch tâm
        hd = calc_hd95(pm.astype(bool), gm.astype(bool))
        cbl = calc_cbl(pm.astype(bool), gm.astype(bool))

        all_dice.append(dice)
        all_iou.append(iou)
        all_pre.append(pre)
        all_rec.append(rec)
        all_hd95.append(hd)
        if cbl is not None:
            all_cbl.append(cbl)

        # =====================================================
        # TRỰC QUAN HÓA (4 CỘT - GIỐNG EXP A/B)
        # =====================================================
        if idx in SHOW_INDEX:
            cx_gt, cy_gt = get_centroid(gm)
            cx_p, cy_p   = get_centroid(pm)

            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(f"Kiểm thử Attention U-Net | Sample ID: {idx}", fontsize=14, fontweight='bold')

            # 1. Ảnh gốc
            axes[0].imshow(img_np, cmap='gray')
            axes[0].set_title("Ảnh gốc", fontsize=11, fontweight='bold')

            # 2. Ground Truth + Tâm
            axes[1].imshow(img_np, cmap='gray')
            green = np.zeros((*gm.shape, 4), dtype=np.float32)
            green[gm == 1] = [0, 1, 0, 0.35]
            axes[1].imshow(green)
            if gm.max() > 0:
                axes[1].contour(gm, [0.5], colors='lime', linewidths=1.5)
            if cx_gt is not None:
                axes[1].plot(cx_gt, cy_gt, 'o', color='lime', ms=8, markeredgecolor='black', label='Tâm GT')
                axes[1].legend(loc='lower right', fontsize=8)
            axes[1].set_title("Ground Truth", fontsize=11, fontweight='bold')

            # 3. Dự đoán + Tâm
            axes[2].imshow(img_np, cmap='gray')
            red = np.zeros((*pm.shape, 4), dtype=np.float32)
            red[pm == 1] = [1, 0, 0, 0.35]
            axes[2].imshow(red)
            if pm.max() > 0:
                axes[2].contour(pm, [0.5], colors='red', linewidths=1.5)
            if cx_p is not None:
                axes[2].plot(cx_p, cy_p, 'o', color='red', ms=8, markeredgecolor='white', label='Tâm Pred')
                axes[2].legend(loc='lower right', fontsize=8)
            axes[2].set_title("Dự đoán (Att-UNet)", fontsize=11, fontweight='bold')

            # 4. Nét cắt so sánh + Chỉ số (Có đường nối tâm)
            axes[3].imshow(img_np, cmap='gray')
            if gm.max() > 0: axes[3].contour(gm, [0.5], colors='lime', linewidths=2)
            if pm.max() > 0: axes[3].contour(pm, [0.5], colors='red', linewidths=2, linestyles='--')
            if cx_gt is not None: axes[3].plot(cx_gt, cy_gt, 'o', color='lime', ms=8, markeredgecolor='black')
            if cx_p is not None:  axes[3].plot(cx_p, cy_p, 'o', color='red', ms=8, markeredgecolor='white')

            # Vẽ đường nối 2 tâm để hiển thị độ lệch
            if cx_gt is not None and cx_p is not None:
                axes[3].plot([cx_gt, cx_p], [cy_gt, cy_p], '--', color='yellow', lw=1.5)

            title_str = (f"Dice: {dice:.3f} | IoU: {iou:.3f} | HD95: {hd:.1f}px\n"
                         f"CBL: {cbl:.3f} | Pre: {pre:.3f} | Rec: {rec:.3f}")
            axes[3].set_title(title_str, fontsize=10, fontweight='bold')

            for ax in axes: ax.axis('off')
            plt.tight_layout()
            plt.show()

# =========================================================
# FINAL REPORT (TỔNG HỢP CHO LUẬN VĂN)
# =========================================================
print("\n" + "=" * 60)
print("📊 FINAL TEST RESULTS - ATTENTION U-NET")
print("=" * 60)
print(f"Mean Dice ↑      : {np.mean(all_dice):.4f}")
print(f"Mean IoU ↑       : {np.mean(all_iou):.4f}")
print(f"Mean Precision ↑ : {np.mean(all_pre):.4f}")
print(f"Mean Recall ↑    : {np.mean(all_rec):.4f}")
print(f"Mean HD95 ↓ (px) : {np.mean(all_hd95):.2f}")
print(f"Mean CBL ↑       : {np.mean(all_cbl):.4f}")
print(f"Total Samples    : {len(all_dice)}")
print("=" * 60)
# =========================================================
# LUU KET QUA CSV (LUAN VAN)
# =========================================================
import csv, os
os.makedirs("results", exist_ok=True)
csv_path = "results/attunet2d_results.csv"
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["model", "dice", "iou", "precision", "recall", "hd95", "cbl", "n_samples"])
    writer.writerow([
        "AttUNet2D",
        f"{np.mean(all_dice):.4f}",
        f"{np.mean(all_iou):.4f}",
        f"{np.mean(all_pre):.4f}",
        f"{np.mean(all_rec):.4f}",
        f"{np.mean(all_hd95):.4f}",
        f"{np.mean(all_cbl):.4f}",
        len(all_dice)
    ])
print(f"\nKet qua da luu: {csv_path}")

# =========================================================
# LUU KET QUA PER-IMAGE CSV (phuc vu kiem dinh thong ke neu can)
# =========================================================
img_names = [os.path.basename(p) for p in test_dataset.images]
per_image_path = "results/attunet2d_per_image.csv"
with open(per_image_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["img_name", "dice", "iou", "precision", "recall", "hd95"])
    for i in range(len(all_dice)):
        writer.writerow([
            img_names[i] if i < len(img_names) else f"idx_{i}",
            f"{all_dice[i]:.6f}", f"{all_iou[i]:.6f}",
            f"{all_pre[i]:.6f}", f"{all_rec[i]:.6f}", f"{all_hd95[i]:.6f}"
        ])
print(f"Ket qua per-image da luu: {per_image_path}")
